# Mid-Term Practical Evaluation — Machine Learning
**Universidad Internacional del Ecuador — Escuela de Ciencias de la Computación**
**Autor:** Daniel Sozoranga · **Docente:** Ing. Jonathan E. Tito O., MSc.
**Duration:** 90 min · **Individual** · **Total:** 12 pts

---

---
# PART A — Setup & Diagnostics (2.5 pts, ~20 min)

## Task 1 — Environment check (0.5 pts)
Run the cell below. If you see version numbers printed, you are ready.

In [ ]:
import sys
import pandas as pd
import sklearn
import numpy as np

print(f"Python ......... {sys.version.split()[0]}")
print(f"pandas ......... {pd.__version__}")
print(f"scikit-learn ... {sklearn.__version__}")
print(f"numpy .......... {np.__version__}")
print("\nEnvironment OK — you can proceed.")

## Task 2 — Fix the bug in `src/scale.py` (1 pt)

**Bug encontrado:** el numerador decía `arr - col_max` en lugar de `arr - col_min`.
- Con `col_max`: cuando `x = col_min` → `(col_min - col_max)/(col_max - col_min) = -1` ❌
- Con `col_min`: cuando `x = col_min` → `0 / rango = 0` ✅

**Fix aplicado en `src/scale.py` (línea del return):**
```python
# BUG: el numerador usaba col_max en lugar de col_min, produciendo rango [-1, 0].
# FIX: reemplazar col_max por col_min para obtener rango [0, 1].
return (arr - col_min) / (col_max - col_min)
```

In [ ]:
# Verification cell — do not modify
import importlib
from src import scale
importlib.reload(scale)
import numpy as np

test_data = np.array([10, 20, 30, 40, 50])
scaled = scale.min_max_scale(test_data)

assert scaled.min() == 0.0, f"min should be 0, got {scaled.min()}"
assert scaled.max() == 1.0, f"max should be 1, got {scaled.max()}"
print(f"Scaled output: {scaled}")
print("Bug fix verified: OK")

## Task 3 — Pandas indexing (1 pt)

In [ ]:
import pandas as pd

df_small = pd.DataFrame({
    "alcohol":   [9.5, 10.2, 11.0, 12.8, 9.9],
    "ph":        [3.4, 3.2, 3.5, 3.1, 3.3],
    "category":  pd.Categorical(["train", "train", "test", "test", "train"]),
})
df_small

In [ ]:
# Task 3a: dtype de la columna 'category'
print(df_small["category"].dtype)
# Es CategoricalDtype (NO 'object') porque se creó con pd.Categorical().
# Pandas almacena los valores como códigos enteros + índice de categorías,
# lo que reduce memoria y permite ordenamiento semántico por categoría.

# Task 3b: label-based indexing con .loc — usa el LABEL del índice, no la posición
# El índice por defecto es 0,1,2,3,4 → label 3 = cuarta fila → alcohol = 12.8
target_value = df_small.loc[3, "alcohol"]
print(f"target_value = {target_value}")

---
# PART B — Linear Regression (5 pts, ~35 min)

## Load the dataset
Run the cell below. **Do not modify it.**

In [ ]:
import pandas as pd
df = pd.read_csv("../data/wine_quality.csv")
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## Task 4 — Quick EDA (1 pt)
Dos plots: distribución del target `quality` + heatmap de correlación.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: distribución del target
quality_counts = df["quality"].value_counts().sort_index()
axes[0].bar(quality_counts.index.astype(str), quality_counts.values,
            color="steelblue", edgecolor="white")
axes[0].set_xlabel("Calidad (quality)")
axes[0].set_ylabel("Número de muestras")
axes[0].set_title("Distribución del target: quality")
for i, (q, v) in enumerate(quality_counts.items()):
    axes[0].text(i, v + 3, str(v), ha="center", fontsize=9, fontweight="bold")

# Plot 2: heatmap de correlación
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            ax=axes[1], cbar_kws={"shrink": 0.8})
axes[1].set_title("Mapa de correlación (Pearson)")

plt.tight_layout()
plt.show()

**Observaciones:**

- **(a)** La feature con mayor correlación con `quality` es **`alcohol`** (r ≈ 0.36): a mayor contenido alcohólico, mayor puntuación de calidad.
- **(b)** El dataset presenta **desbalance de clases severo**: las calidades extremas (3 → 10 muestras, 8 → 18 muestras) tienen muy pocas instancias frente a las centrales (5 → 681, 6 → 638), lo que puede sesgar el modelo a predecir valores medios.

## Task 5 — Preprocessing Pipeline (2 pts)

**Justificación del scaler:** uso `StandardScaler` porque las features tienen escalas muy distintas (`total_sulfur_dioxide` ~ 46 vs `citric_acid` ~ 0.27). Estandarizar a media=0, std=1 hace los coeficientes comparables y estabiliza el descenso de gradiente. `MinMaxScaler` sería sensible a los outliers presentes en el dataset.

**El Pipeline se fittea SOLO en `X_train`** — el scaler aprende μ y σ únicamente de train y los aplica a test vía `.transform()`, evitando data leakage.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 1. X e y
X = df.drop(columns=["quality"])
y = df["quality"]

# 2. Split 80/20, random_state=42 fijo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 3. Pipeline: StandardScaler (justificado arriba) + LinearRegression
pipe_linear = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model",  LinearRegression()),
])

# 4. Fit SOLO sobre training set — el scaler NO ve X_test
pipe_linear.fit(X_train, y_train)

print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")
print(f"Pasos del pipeline: {[name for name, _ in pipe_linear.steps]}")

## Task 6 — Train & evaluate linear regression (2 pts)

In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y_pred = pipe_linear.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 45)
print("LINEAR REGRESSION — Test set metrics")
print("=" * 45)
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : {mae:.4f}  (puntos de calidad)")
print(f"  RMSE : {rmse:.4f}")

**Interpretación:**

El modelo explica solo el **31.6% de la varianza** del target (R² ≈ 0.32), con un error promedio de **±0.54 puntos** (MAE); en una escala de 3 a 8, esto indica que la relación fisicoquímica→calidad existe pero es débil, haciendo al modelo poco usable para predicciones precisas — el error típico equivale a confundir un vino de calidad 5 con uno de 6.